In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

In [ ]:
class AdaIN(nn.Module):
    def __init__(self):
        super(AdaIN, self).__init__()
    def calc_mean_std(self, feat, eps=1e-5):
        # feat: [batch, channel, height, width]
        size = feat.size()
        assert (len(size) == 4)
        N, C = size[:2]
        feat_var = feat.view(N, C, -1).var(dim=2) + eps
        feat_std = feat_var.sqrt().view(N, C, 1, 1)
        feat_mean = feat.view(N, C, -1).mean(dim=2).view(N, C, 1, 1)
        return feat_mean, feat_std
    def forward(self, content_feat, style_feat):
        # Lấy thống kê của nội dung và phong cách
        content_mean, content_std = self.calc_mean_std(content_feat)
        style_mean, style_std = self.calc_mean_std(style_feat)
        # Chuẩn hóa nội dung và áp phong cách mới
        normalized_feat = (content_feat - content_mean) / content_std
        return normalized_feat * style_std + style_mean

In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super(Encoder, self).__init__()
        vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features
        # Cắt lấy các tầng cần thiết: từ 0 đến 21 (vượt qua ReLU 4_1)
        self.encoder = nn.Sequential(*list(vgg.children())[:21])
        # Đóng băng tham số vgg
        for param in self.parameters():
            param.requires_grad = False
    def forward(self, x):
        return self.encoder(x)

        
# 3. Xây dựng Decoder (Đối xứng với Encoder nhưng dùng Upsampling thay vì Pooling)
def rc_block(in_planes, out_planes):
    return nn.Sequential(
        nn.ReflectionPad2d(1),
        nn.Conv2d(in_planes, out_planes, kernel_size=3),
        nn.ReLU(inplace=True)
    )

class Decoder(nn.Module):
    def __init__(self):
        super(Decoder, self).__init__()
        # Decoder đi từ đặc trưng relu4_1 (512 kênh) về ảnh gốc (3 kênh)
        self.decoder = nn.Sequential(
            rc_block(512, 256),
            nn.Upsample(scale_factor=2, mode='nearest'),
            rc_block(256, 256),
            rc_block(256, 256),
            rc_block(256, 256),
            rc_block(256, 128),
            nn.Upsample(scale_factor=2, mode='nearest'),
            rc_block(128, 128),
            rc_block(128, 64),
            nn.Upsample(scale_factor=2, mode='nearest'),
            rc_block(64, 64),
            nn.ReflectionPad2d(1),
            nn.Conv2d(64, 3, kernel_size=3) # Output: 3 channels
        )
    def forward(self, x):
        return self.decoder(x)

In [ ]:
# 4. Mạng Style Transfer hoàn chỉnh
class AdaINNet(nn.Module):
    def __init__(self):
        super(AdaINNet, self).__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()
        self.adain = AdaIN()
    def forward(self, content, style, alpha=1.0):
        content_feat = self.encoder(content)
        style_feat = self.encoder(style)
        
        # Áp dụng AdaIN
        t = self.adain(content_feat, style_feat)
        
        # Alpha dùng để điều chỉnh cường độ style (0: nội dung gốc, 1: phong cách đầy đủ)
        t = alpha * t + (1 - alpha) * content_feat
        
        # Giải mã về ảnh
        return self.decoder(t)
# 5. Hàm Loss (Theo Keras example)
def calc_content_loss(input_feat, target_feat):
    return F.mse_loss(input_feat, target_feat)
def calc_style_loss(input_feat, target_feat):
    adain = AdaIN()
    # Style loss dựa trên sai số của Mean và Std giữa layer ảnh gen và ảnh style
    input_mean, input_std = adain.calc_mean_std(input_feat)
    target_mean, target_std = adain.calc_mean_std(target_feat)
    
    loss_mean = F.mse_loss(input_mean, target_mean)
    loss_std = F.mse_loss(input_std, target_std)
    return loss_mean + loss_std
# --- Cách sử dụng ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AdaINNet().to(device)
# Trong quá trình Training:
# 1. Lấy ảnh gen: generated_img = model(content_img, style_img)
# 2. Lấy đặc trưng ảnh gen: gen_feat = model.encoder(generated_img)
# 3. Tính Content Loss giữa gen_feat và output của AdaIN (t).
# 4. Tính Style Loss giữa gen_feat và style_feat trên nhiều layers.